In [1]:
!python -m pip uninstall torch torchvision torchaudio -y

In [2]:
!conda remove pytorch torchvision torchaudio -y

'conda' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download-r2.pytorch.org/whl/cpu/torch-2.12.0%2Bcpu-cp311-cp311-win_amd64.whl.metadata (32 kB)
  Using cached https://download-r2.pytorch.org/whl/cpu/torchvision-0.27.0%2Bcpu-cp311-cp311-win_amd64.whl.metadata (5.6 kB)
Using cached https://download-r2.pytorch.org/whl/cpu/torch-2.12.0%2Bcpu-cp311-cp311-win_amd64.whl (122.9 MB)
Using cached https://download-r2.pytorch.org/whl/cpu/torchvision-0.27.0%2Bcpu-cp311-cp311-win_amd64.whl (3.7 MB)

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
!python -m pip install timm

In [5]:
import torch

midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")

transform = midas_transforms.small_transform

Using cache found in C:\Users\Hamza/.cache\torch\hub\intel-isl_MiDaS_master
c:\Users\Hamza\anaconda3\envs\sit332\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Hamza\anaconda3\envs\sit332\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None


Using cache found in C:\Users\Hamza/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\Hamza/.cache\torch\hub\intel-isl_MiDaS_master


# 9th May 2026

In [6]:
import torch

midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")

transform = midas_transforms.small_transform

Using cache found in C:\Users\Hamza/.cache\torch\hub\intel-isl_MiDaS_master
Using cache found in C:\Users\Hamza/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master


Loading weights:  None


Using cache found in C:\Users\Hamza/.cache\torch\hub\intel-isl_MiDaS_master


## Methods for Path Detection

In [7]:
!python -m pip uninstall opencv-python -y 
!python -m pip install opencv-python

Found existing installation: opencv-python 4.12.0


error: uninstall-no-record-file

× Cannot uninstall opencv-python 4.12.0
╰─> The package's contents are unknown: no RECORD file was found for opencv-python.

hint: The package was installed by conda. You should check if it can uninstall the package.


## Scene Understanding

## Terrain Awareness

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch

midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.small_transform

import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import time

# =========================
# CONFIG
# =========================
USE_OUTPUT = True
OUTPUT_PATH = "output.mp4"

MAX_DISTANCE = 1000
MAX_SPEED = 50
GROUND_RATIO = 0.65

IMPORTANT_CLASSES = {"person", "car", "bicycle", "motorcycle"}

priority_map = {
    "person": 1.0,
    "car": 0.9,
    "bicycle": 0.8,
    "motorcycle": 0.85,
    "dog": 0.6,
}

# =========================
# MODEL
# =========================
model = YOLO("yolo11n.pt")
track_history = defaultdict(list)

# =========================
# FUNCTIONS (UNCHANGED)
# =========================
def compute_ttc(distance, speed):
    return float("inf") if speed < 1 else distance / speed

def compute_risk(obj_class, distance, speed, position_score, track, center):
    priority = priority_map.get(obj_class, 0.1)
    proximity = distance
    speed_norm = min(speed / 50.0, 1.0)

    alignment = 0.0
    if len(track) >= 3:
        p_old = np.array(track[-3], dtype=np.float32)
        p_now = np.array(track[-1], dtype=np.float32)

        velocity = p_now - p_old
        to_center = center - p_now

        if np.linalg.norm(velocity) > 1e-3:
            alignment = np.dot(velocity, to_center) / (
                np.linalg.norm(velocity) * np.linalg.norm(to_center) + 1e-6
            )
            alignment = max(alignment, 0)

    ttc = (distance + 1e-3) / (speed + 1e-3)
    ttc_score = np.exp(-ttc)

    score = (
        0.35 * proximity +
        0.20 * speed_norm +
        0.20 * alignment +
        0.15 * ttc_score +
        0.10 * position_score
    )

    return priority * score

def risk_level(score):
    if score > 0.75:
        return "HIGH"
    elif score > 0.4:
        return "MEDIUM"
    return "LOW"

def predict_collision(track, center, frame_width):
    if len(track) < 5:
        return False

    PATH_HALF_WIDTH = int((0.2 * frame_width) / 2)

    pts = np.array(track[-5:], dtype=np.float32)
    current = pts[-1]
    velocity = pts[-1] - pts[0]

    if abs(current[0] - center[0]) > PATH_HALF_WIDTH:
        return False
    if velocity[1] < 2:
        return False
    if np.dot(velocity, center - current) <= 0:
        return False
    if np.linalg.norm(velocity) < 5:
        return False

    future = current + velocity * 1.2
    return abs(future[0] - center[0]) <= PATH_HALF_WIDTH

def avoidance_action(level, collision, cx, center_x):
    if not collision:
        return "NONE"
    if level == "HIGH":
        return "STOP"
    return "LEFT" if cx > center_x else "RIGHT"

def get_depth_distance(depth_map, x, y):
    h, w = depth_map.shape
    x = np.clip(x, 0, w - 1)
    y = np.clip(y, 0, h - 1)
    return depth_map[int(y), int(x)]

# =========================
# 🧠 IMPROVED TERRAIN AWARENESS (HANDHELD CAMERA ROBUST)
# =========================

terrain_smooth = None
prev_terrain_vec = None

def detect_terrain_change(depth_map):
    global terrain_smooth, prev_terrain_vec

    h, w = depth_map.shape

    # -------------------------
    # 1. Multi-strip ground sampling (robust to shake)
    # -------------------------
    strip1 = depth_map[int(0.55*h):int(0.60*h), :]
    strip2 = depth_map[int(0.60*h):int(0.65*h), :]
    strip3 = depth_map[int(0.65*h):int(0.70*h), :]

    profile = np.concatenate([
        np.mean(strip1, axis=0),
        np.mean(strip2, axis=0),
        np.mean(strip3, axis=0),
    ])

    # -------------------------
    # 2. Temporal smoothing (removes walking jitter)
    # -------------------------
    if terrain_smooth is None:
        terrain_smooth = profile
    else:
        terrain_smooth = 0.85 * terrain_smooth + 0.15 * profile

    # -------------------------
    # 3. Motion normalization (camera shake removal)
    # -------------------------
    diff = np.diff(terrain_smooth)

    mean_change = np.mean(diff)
    std_change = np.std(diff)

    # -------------------------
    # 4. Walking motion suppression
    # -------------------------
    if std_change > 0.10:
        return "WALKING / CAMERA MOTION"

    # -------------------------
    # 5. Terrain classification (stable signals only)
    # -------------------------
    if mean_change > 0.04:
        return "UPHILL / STEP UP AHEAD"
    elif mean_change < -0.04:
        return "DOWNHILL / DROP AHEAD"
    elif std_change > 0.06:
        return "ROUGH / UNEVEN GROUND"
    else:
        return "FLAT GROUND"

# =========================
# VIDEO INPUT
# =========================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Cannot access webcam")

FRAME_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30

CENTER = np.array([FRAME_WIDTH // 2, FRAME_HEIGHT // 2], dtype=np.float32)

if USE_OUTPUT:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (FRAME_WIDTH, FRAME_HEIGHT))

# =========================
# MAIN LOOP
# =========================
prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = transform(img_rgb).to(device)

    with torch.no_grad():
        depth = midas(input_batch)
        depth = torch.nn.functional.interpolate(
            depth.unsqueeze(1),
            size=frame.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    depth_map = depth.cpu().numpy()
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-6)
    depth_map = cv2.GaussianBlur(depth_map, (5, 5), 0)

    # =========================
    # TERRAIN (IMPROVED)
    # =========================
    terrain_state = detect_terrain_change(depth_map)

    results = model.track(frame, persist=True, device="cpu", verbose=False)

    detected_objects = 0
    scene_mode = "CLEAR PATH"

    if results and results[0].boxes is not None and results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        detected_objects = len([c for c in classes if model.names[int(c)] in IMPORTANT_CLASSES])

        if detected_objects >= 8:
            scene_mode = "CROWDED SCENE"
        elif detected_objects >= 4:
            scene_mode = "MODERATE OBSTACLES"
        else:
            scene_mode = "CLEAR PATH"

        for box, track_id, cls in zip(boxes, ids, classes):

            obj_name = model.names[int(cls)]
            if obj_name not in IMPORTANT_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box)

            foot_x = (x1 + x2) // 2
            foot_y = y2

            track = track_history[track_id]
            track.append((foot_x, foot_y))
            if len(track) > 20:
                track.pop(0)

            raw_depth = get_depth_distance(depth_map, foot_x, foot_y)
            distance = 1 - raw_depth

            speed = 0
            if len(track) > 1:
                dx = track[-1][0] - track[-2][0]
                dy = track[-1][1] - track[-2][1]
                speed = np.sqrt(dx**2 + dy**2)

            pos_dist = np.linalg.norm(np.array([foot_x, foot_y]) - CENTER)
            position_score = 1 - min(pos_dist / np.linalg.norm(CENTER), 1)

            score = compute_risk(obj_name, distance, speed, position_score, track, CENTER)
            level = risk_level(score)

            color = (0, 255, 0)
            if level == "MEDIUM":
                color = (0, 255, 255)
            elif level == "HIGH":
                color = (0, 0, 255)

            label = f"{obj_name} | {level}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # =========================
    # DISPLAY
    # =========================
    cv2.putText(frame, f"Scene: {scene_mode}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Terrain: {terrain_state}", (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)

    cv2.imshow("RGB View", frame)

    if USE_OUTPUT:
        out.write(frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
if USE_OUTPUT:
    out.release()
cv2.destroyAllWindows()

print("Done.")

Using cache found in C:\Users\Hamza/.cache\torch\hub\intel-isl_MiDaS_master
Using cache found in C:\Users\Hamza/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master


Loading weights:  None


Using cache found in C:\Users\Hamza/.cache\torch\hub\intel-isl_MiDaS_master
